# Notes

<u>System questions/notes:</u>
* What is a cluster? Do we need replicas?
* Collection - do we need partitions?
* The schema seems automatically determined based on the attributes and fields of the Documents. AutoID and dynamic field are N/A (`False` and `False`) when using Milvus with Haystack.
    ```python
    assert doc.id == res[0]["id"] # primary field
    assert doc.embedding == res[0]["vector"] # embedding
    assert doc.content == res[0]["text"] # text
    assert doc.meta["source_id"] == res[0]["source_id"] # autogenerated file ID
    assert doc.meta["page_number"] == res[0]["page_number"] # file page number
    assert doc.meta["split_id"] == res[0]["split_id"] # split index
    assert doc.meta["split_idx_start"] == res[0]["split_idx_start"] # not sure what this means
    assert doc.meta["file_path"] == res[0]["file_path"] # file
    ```
    * Primary field needs to be unique, but if I'm adding documents to an already existing collection, how do I know "id" and "source_id" will be unique?
    * doc.meta["_split_overlap"] was discarded with the error `has metadata fields with unsupported types: ['_split_overlap']. Supported types refer to Pymilvus DataType. The values of these fields will be discarded.`. I think this is because doc.meta["_split_overlap"] is `list[dict]`. This might be important for retrieval.
    * Understand what the metadata means and how it's being generated.
    * Is there any additional metadata that we might want to add that isn't autogenerated? Can we add additional metadata in the pipeline? Or do we need to predefine metadata in the vector DB, before using the pipeline to write to it? Example metadata:
        * **tag**
        * Course / course number / course name
        * Lecturer
        * Year / semester
        * Lecture number or lecture title
    * Is it possible for different file types to have different metadata? If so, how can we handle that?
* Figure out how AUTOINDEX works; understand all the different indexes and metrics
    * What kind of metric should we use? This depends on whether the SentenceTransformer embeddings are normalized. What's the best practice?
    * Can I change the metric after the collection is created? I'm guessing that yes, I can, but the indexing will have to be rerun since the indexing depends on the metric.
    * You should also index by tag. Zilliz uses **TRIE** for integers and **STL_SORT** for strings. Can I add an index after the collection is already created?
* Vector fields
    * Should we have any kind of metadata embedding? https://haystack.deepset.ai/tutorials/39_embedding_metadata_for_improved_retrieval
    * Multimodal embedding for images?
    * Utilize both dense and sparse embeddings, and search both using `hybrid_search()`?
* Pipelines
    * Indexing - this should be relatively straightforward
    * Tagging - get the new IDs that were indexed into the vector DB; pass them through the LLM to generate tags; add tag metadata to the new entities

<u>General questions:</u>
* If we know the course the learning material is from, why can't we get the tag directly based on the course/learning track? Guess: Courses may cover many subdisciplines; courses are not necessarily rigidly only one discipline.

<u>Zilliz notes:</u>
* Quickstart notes
    * The insert operations are asynchronous, and conducting a search immediately after data insertions may result in empty result set. To avoid this, you are advised to wait for a few seconds.
    * Searches are semantic searches (client.search), but you can also apply scalar field filters. Queries (client.query) are based on scalar filters only. You can also directly retrieve entities by their ID using client.get (instead of using query).
* Collection notes
    * You need to load a collection into memory to search and query it. This means loading the index files and the raw data of the fields.
    * A collection cannot be loaded without an index file
    * To reduce memory usage and improve search performance, you can specify which fields you want to load (instead of all fields)
        * Only these fields may be used for filtering and as outputs in search and query
        * You should always include the primary field and at least one vector field
    * Entities inserted after a collection load are automatically indexed and loaded
* Indexing notes
    * Recommended to create indexes for both vector field(s) and scalar field(s) that are frequently accessed.
    * Vector field indexes are for semantic search; scalar field indexes are for metadata filtering.
    * You can create up to one index per field in a collection.
    * You can always modify indexes by dropping the old index and adding a new one
    * For vector field indexes, Zilliz Cloud supports AUTOINDEX (https://docs.zilliz.com/docs/autoindex-explained)
        * Performance-optimized and capacity-optimized clusters require different approaches to indexing - AUTOINDEX takes care of that
        * Improved performance via SIMD, data graphing and cropping, and dynamic quantization
        * AUTOINDEX automatically chooses search parameters to trade off between recall and performance. Search params only has 1 parameter: level. Higher level = higher recall, but possibly slower search. Level defaults to 1 and ranges from 1 to 10. Default value = 90% recall. `enable_recall_calculation`?

<u>Action items:</u>
* Add tag as a field to the schema - `DataType.ARRAY` makes sense for this
    * If you define a schema field that isn't automatically generated by the Haystack pipeline, then a KeyError exception is thrown when you run the pipeline - in MilvusDocumentStore.write_documents, there is a line of code that iterates through the fields `insert_list = [insert_dict[x][i:end] for x in self.fields]`. `self.fields` is initialized via `_extract_fields()` that extracts fields from the collection schema, which includes `tags`, but since the pipeline doesn't generate a `tags` field, there is a KeyError.
    * I think you need to include `tags` when creating the collection. You can create it after, when you actually do the tagging, but you'll just run into the same exception since new documents will pass through the indexing pipeline.
    * Setting to nullable doesn't help since it's still part of the schema
    * Create a custom component to add this metadata to Document
    * There seems to be some issue with MilvusDocumentStore calling `pymilvus.orm.types.infer_dtype_bydata`. For `list[str]`, this returns `DataType.UNKNOWN` but should return `DataType.ARRAY`.
        * I am able to insert `list[str]` as an ARRAY using pymilvus, but this doesn't work with Haystack. `infer_dtype_bydata` returns `UNKNOWN` for `list[str]` and `FLOAT_VECTOR` for `list[int]`, neither of which are correct. For `list[int]`, creating the collection fails because it thinks that `"tags"` is a vector, and there's no `dim` param:
        ```python
        {'name': 'tags', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>}
        {'name': 'tags', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>}
        {'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 65535}}
        {'name': 'id', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 65535}, 'is_primary': True, 'auto_id': False}
        {'name': 'vector', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 3}}
        2025-02-25 08:44:46,884 [ERROR][handler]: RPC error: [create_collection], <MilvusException: (code=65535, message=dimension is not defined in field type params, check type param `dim` for vector field)>, <Time:{'RPC start': '2025-02-25 08:44:46.799552', 'RPC error': '2025-02-25 08:44:46.883781'}> (decorators.py:140)
        Failed to create collection: TestARRAYInsert error: <MilvusException: (code=65535, message=dimension is not defined in field type params, check type param `dim` for vector field)>
        ```
    * Try using a JSON object
    * Create a custom component to wrap around pymilvus
* Check if `split_overlap` doc IDs match document IDs in the vector store. Can you convert this to a JSON object or an ARRAY? For JSON, perhaps you can convert from list of dicts to just one dict.
* How does Haystack determine document and source IDs? Are these guaranteed to be unique?

# Zilliz/Milvus API

In [ ]:
from pymilvus import MilvusClient, DataType
from dotenv import load_dotenv
import os

load_dotenv()

In [ ]:
client = MilvusClient(
    uri=os.getenv("ZILLIZ_CLUSTER_ENDPOINT"),
    token=os.getenv("ZILLIZ_CLUSTER_TOKEN")
)

# client = MilvusClient(
#     uri="milvus.db"
# )

In [ ]:
client.close()

## Collections

In [ ]:
# View collections

coll_name = client.list_collections()[0]
# print(coll_name)
# print("-"*20)
coll_description = client.describe_collection(coll_name)
for key, value in coll_description.items():
    if type(value) == list:
        print(f"{key}:")
        for v in value:
            print(v)
    else:
        print(f"{key}: {value}")
    print("-"*20)

In [ ]:
# Load collection into memory for search and query

client.load_collection(coll_name)
client.get_load_state(coll_name)

In [ ]:
# Release collection from memory

client.release_collection(coll_name)
client.get_load_state(coll_name)

## Indexes

In [ ]:
# List and describe indexes

index_name = client.list_indexes(coll_name)[0]
client.describe_index(coll_name, index_name)

In [ ]:
# Drop index

client.release_collection(coll_name)
client.get_load_state(coll_name)
client.drop_index(coll_name, index_name)
client.list_indexes(coll_name)

In [ ]:
# Add index

index_params = client.prepare_index_params()

index_params.add_index(
    field_name = "vector",
    metric_type = "L2",
    index_type = "AUTOINDEX",
    index_name = "vector_index"
)

client.create_index(coll_name, index_params)

# Haystack-Zilliz Indexing Pipeline

In [1]:
from haystack import Pipeline, Document, component
from milvus_haystack import MilvusDocumentStore
from haystack.components.converters import PyPDFToDocument
from haystack.components.preprocessors import DocumentCleaner, DocumentSplitter
from haystack.components.embedders import SentenceTransformersDocumentEmbedder
from haystack.components.writers import DocumentWriter
from haystack.utils import Secret
from pymilvus import MilvusClient, DataType
from typing import List

In [ ]:
client = MilvusClient(
    uri = Secret.from_env_var("ZILLIZ_CLUSTER_ENDPOINT").resolve_value(),
    token = Secret.from_env_var("ZILLIZ_CLUSTER_TOKEN").resolve_value(),
)

In [ ]:
# https://docs.zilliz.com/docs/manage-collections-sdks
# https://milvus.io/docs/array_data_type.md
# https://milvus.io/docs/dense-vector.md

schema = MilvusClient.create_schema(
    auto_id = False, # how does Haystack generate unique IDs?
    enable_dynamic_field = False,
)

schema.add_field(field_name="id", datatype=DataType.VARCHAR, is_primary=True, auto_id=False, max_length=512)
schema.add_field(field_name="vector", datatype=DataType.FLOAT_VECTOR, dim=768)
# schema.add_field(field_name="tags", datatype=DataType.ARRAY, element_type=DataType.VARCHAR, max_capacity=10, max_length=20, nullable=True)
# schema.add_field(field_name="split_overlap_ids", datatype=DataType.ARRAY, element_type=DataType.VARCHAR, max_capacity=2, max_length=512, nullable=True)

index_params = client.prepare_index_params()

index_params.add_index(
    field_name = "vector",
    metric_type = "COSINE",
    index_type = "AUTOINDEX",
    index_name = "vector_index"
)

# You can specify other index types
# index_params.add_index(
#     field_name = "tags",
#     index_type = "AUTOINDEX",
#     index_name = "tags_index",
# )

if "HaystackCollection" in client.list_collections():
    client.drop_collection("HaystackCollection")
    
client.create_collection(
    collection_name = "HaystackCollection",
    schema = schema,
    index_params = index_params,
)

client.get_load_state("HaystackCollection")

In [ ]:
# Connect to Milvus client and create new collection

file_names = ["Project Management Requirements Handbook.pdf"]

document_store = MilvusDocumentStore(
    collection_name = "HaystackCollection",
    collection_description = "Test collection",
    collection_properties = None,
    connection_args = {
        # "uri": "https://in01-65d96e8e6d6e34c.aws-us-east-1.vectordb.zillizcloud.com:19539",  # Public Endpoint
        "uri": Secret.from_env_var("ZILLIZ_CLUSTER_ENDPOINT").resolve_value(),
        "token": Secret.from_env_var("ZILLIZ_CLUSTER_TOKEN").resolve_value(),  # API key.
        "secure": True
        },
    consistency_level = "Strong", # Strong, Bounded, Eventually, Session
    index_params = {
        "index_type": "AUTOINDEX",
        "metric_type": "COSINE",
    },
    search_params = {
        "params": {
            "level": 1
        }
    },
    drop_old = True,
)

# Connect to Milvus client and add to an existing collection

# file_names = ["Project Management Requirements Handbook.pdf", "Lec1 Machine Learning Review.pdf"]

# document_store = MilvusDocumentStore(
#     collection_name = "HaystackCollection",
#     # collection_description = "Test collection",
#     # collection_properties = None,
#     connection_args = {
#         # "uri": "https://in01-65d96e8e6d6e34c.aws-us-east-1.vectordb.zillizcloud.com:19539",  # Public Endpoint
#         "uri": Secret.from_env_var("ZILLIZ_CLUSTER_ENDPOINT").resolve_value(),
#         "token": Secret.from_env_var("ZILLIZ_CLUSTER_TOKEN").resolve_value(),  # API key.
#         "secure": True
#         },
#     # consistency_level = "Strong", # Strong, Bounded, Eventually, Session
#     # index_params = {
#     #     "index_type": "AUTOINDEX",
#     #     "metric_type": "COSINE",
#     # },
#     # search_params = {
#     #     "params": {
#     #         "level": 1
#     #     }
#     # },
#     # drop_old=False,
# )

In [ ]:
# Create custom component to handle _split_overlap and to add null tags

add_MetadataCleaner = False

@component
class MetadataCleaner:
    @component.output_types(docs=List[Document])
    def run(self, docs: List[Document]):
        docs = [self._add_metadata(doc) for doc in docs]
        return {"docs": docs}
    
    def _add_metadata(self, doc: Document) -> Document:
        # doc.meta["tags"] = ["None"]
        doc.meta["_split_overlap"] = [d["doc_id"] for d in doc.meta["_split_overlap"]]
        # doc.meta["split_overlap_ids"] = [d["doc_id"] for d in doc.meta["_split_overlap"]]
        return doc

In [ ]:
# Create indexing pipeline

pipe = Pipeline()

pipe.add_component("converter", PyPDFToDocument(extraction_mode="layout"))
pipe.add_component("cleaner", DocumentCleaner())
pipe.add_component("splitter", DocumentSplitter(split_by="word", split_length=100, split_overlap=10, split_threshold=50))
if add_MetadataCleaner:
    pipe.add_component("metadata_cleaner", MetadataCleaner())
pipe.add_component("embedder", SentenceTransformersDocumentEmbedder())
pipe.add_component("writer", DocumentWriter(document_store=document_store))

pipe.connect("converter", "cleaner")
pipe.connect("cleaner", "splitter")
if add_MetadataCleaner:
    pipe.connect("splitter", "metadata_cleaner")
    pipe.connect("metadata_cleaner", "embedder")
else:
    pipe.connect("splitter", "embedder")
pipe.connect("embedder", "writer")

In [ ]:
# Run indexing pipeline

results = pipe.run({"converter": {"sources": file_names}}, include_outputs_from={"embedder"})

In [ ]:
print(document_store.count_documents())
print(document_store.fields)

In [ ]:
# Sanity-check embedder output

idx = 1
doc = results["embedder"]["documents"][idx]
print(doc)
print(f"id: {doc.id}")
for k, v in doc.meta.items():
    print(f"{k}: {v}")

In [4]:
from pymilvus.orm.types import infer_dtype_bydata, is_numeric_datatype, infer_dtype_by_scalar_data
# split_overlap_ids = [d["doc_id"] for d in doc.meta["_split_overlap"]]
split_overlap_ids = ["id1", "id2"]
# split_overlap_ids = [1, 2]
infer_dtype_bydata(split_overlap_ids)
infer_dtype_by_scalar_data(split_overlap_ids)

<DataType.ARRAY: 22>

In [ ]:
from pymilvus.orm.types import infer_dtype_bydata, is_numeric_datatype, infer_dtype_by_scalar_data
from typing import Any

import numpy as np
from pandas.api.types import (
    infer_dtype,
    is_array_like,
    is_float,
    is_list_like,
    is_scalar,
)

from pymilvus.client.types import DataType

dtype_str_map = {
    "string": DataType.VARCHAR,
    "floating": DataType.FLOAT,
    "integer": DataType.INT64,
    "mixed-integer": DataType.INT64,
    "mixed-integer-float": DataType.FLOAT,
    "boolean": DataType.BOOL,
    "mixed": DataType.UNKNOWN,
    "bytes": DataType.UNKNOWN,
}

doc = results["embedder"]["documents"][idx]

print(f"infer_dtype_bydata: {infer_dtype_bydata(doc.meta["split_overlap_ids"])}")
print(f"islistlike: {is_list_like(doc.meta["split_overlap_ids"])}")
type_str = infer_dtype(doc.meta["split_overlap_ids"])
type_str = "integer"
d_type = dtype_str_map.get(type_str, DataType.UNKNOWN)
print(f"typestr: {type_str}")
print(f"dtype: {d_type}")
print(f"isnumeric: {is_numeric_datatype(d_type)}")
print(f"isscalar: {is_scalar(doc.meta["split_overlap_ids"])}")
infer_dtype_by_scalar_data(doc.meta["split_overlap_ids"])

In [ ]:
# infer_dtype_bydata(None)
infer_dtype_by_scalar_data(None)

In [ ]:
# Describe the collection (schema)

col_name = document_store.collection_name

print(document_store.client.get_load_state(col_name))
col_description = document_store.client.describe_collection(col_name)
for k, v in col_description.items():
    if type(v) == list:
        for d in v:
            print(d)
    else:
        print(f"{k}: {v}")

In [ ]:
# Check how the Documents are written into the database (using MilvusClient.query)

expr = "id == {id}"
counter = 0
for doc in results["embedder"]["documents"]:
    filter_params = {"id": doc.id}
    res = document_store.client.query(
        collection_name = col_name,
        filter = expr,
        output_fields = ["*"],
        filter_params = filter_params
    )
    assert len(res) == 1, "Document IDs should map 1:1 to id in the collection"
    assert doc.id == res[0]["id"]
    assert doc.embedding == res[0]["vector"]
    assert doc.content == res[0]["text"]
    # assert doc.meta["source_id"] == res[0]["source_id"]
    # assert doc.meta["page_number"] == res[0]["page_number"]
    # assert doc.meta["split_id"] == res[0]["split_id"]
    # assert doc.meta["split_idx_start"] == res[0]["split_idx_start"]
    # assert doc.meta["file_path"] == res[0]["file_path"]

    for k, v in doc.meta.items():
        if k in res[0]:
            assert doc.meta[k] == res[0][k]

    counter += 1
    print(counter)
    # break

In [ ]:
# Check if doc.meta is the same for all documents (using DocumentStore.filter_documents)
doc = results["embedder"]["documents"][0]

# for field in ["source_id", "page_number", "split_id", "split_idx_start", "file_path"]:
#     expr = f"{field} == " + "{value}"
#     filter_params = {"value": doc.meta[field]}
#     res = document_store.client.query(
#         collection_name = col_name,
#         filter = expr,
#         output_fields = ["*"],
#         filter_params = filter_params
#     )
#     print(len(res))

for field in ["source_id", "page_number", "split_id", "split_idx_start", "file_path"]:
    filters = {"field": f"meta.{field}", "operator": "==", "value": doc.meta[field]}
    # print(filters)
    res = document_store.filter_documents(filters)
    print(len(res))

In [ ]:
# Check doc.meta for all documents

for idx, doc in enumerate(results["embedder"]["documents"]):
    print(idx)
    for k, v in doc.meta.items():
        print(f"{k}: {v}")
    print("-"*30)

In [ ]:
filters = {"field": "meta.source_id", "operator": "==", "value": "cf691c062fc851a2ed51fa786741e86677f0f44e6004c4bff49e5c2dec264f67"}
filters = {"field": "meta.source_id", "operator": "==", "value": "8ac81db781a7877b09869dd6be3c0fdd7a4a737864c9f647bb3cfe3af66cb71e"}
res = document_store.filter_documents(filters)
print(len(res))
# res

# Inserting ARRAY Into Collection
## Using pymilvus

In [ ]:
# https://docs.zilliz.com/docs/use-array-fields

client = MilvusClient(
    uri = Secret.from_env_var("ZILLIZ_CLUSTER_ENDPOINT").resolve_value(),
    token = Secret.from_env_var("ZILLIZ_CLUSTER_TOKEN").resolve_value(),
)

In [ ]:
schema = client.create_schema(
    auto_id = True,
    enable_dynamic_fields = True,
)

schema.add_field(
    field_name = "pk",
    datatype = DataType.VARCHAR,
    max_length = 512,
    is_primary = True
)
schema.add_field(
    field_name = "tags",
    datatype = DataType.ARRAY,
    element_type = DataType.VARCHAR,
    max_capacity = 10,
    max_length = 65535
)
schema.add_field(
    field_name = "vector",
    datatype = DataType.FLOAT_VECTOR,
    dim = 3
)

index_params = client.prepare_index_params()
index_params.add_index(
    field_name = "tags",
    index_type = "AUTOINDEX"
)
index_params.add_index(
    field_name="vector",
    index_type="AUTOINDEX",
    metric_type="COSINE"
)

In [41]:
if "TestARRAYInsert" in client.list_collections():
    client.drop_collection("TestARRAYInsert")
    
client.create_collection(
    collection_name = "TestARRAYInsert",
    schema = schema,
    index_params = index_params
)

In [28]:
data = [
    {"tags": ["pop", "rock"],
     "vector": [0.1, 0.1, 0.1]},
    {"tags": ["pop", "rock"],
     "vector": [0.1, 0.1, 0.1]},
    {"tags": ["pop", "rock"],
     "vector": [0.1, 0.1, 0.1]},
]

In [23]:
client.insert(
    collection_name="TestARRAYInsert",
    data=data,
)

{'insert_count': 3, 'ids': ['456094611417375218', '456094611417375219', '456094611417375220'], 'cost': 0}

## Using Haystack

In [72]:
document_store = MilvusDocumentStore(
    collection_name = "TestARRAYInsert",
    connection_args = {
        "uri": Secret.from_env_var("ZILLIZ_CLUSTER_ENDPOINT").resolve_value(),
        "token": Secret.from_env_var("ZILLIZ_CLUSTER_TOKEN").resolve_value(),
        "secure": True
        },
    drop_old=True,
)

writer = DocumentWriter(document_store=document_store)

In [65]:
class CustomList:
    def __init__(self, data: list):
        self._data = data  # Store internally as a list

    def __getitem__(self, index: int):
        return self._data[index]  # Allow indexing

    def __len__(self) -> int:
        return len(self._data)  # Optional, allows len(data)

In [ ]:
# data = CustomList(["pop", "rock"])

2

In [74]:
# data = [
#     {"id": "1",
#      "meta": {"tags": CustomList(["pop", "rock"])},
#      "embedding": [0.1, 0.1, 0.1]},
#     {"id": "2",
#      "meta": {"tags": CustomList(["pop", "rock"])},
#      "embedding": [0.1, 0.1, 0.1]},
#     {"id": "3",
#      "meta": {"tags": CustomList(["pop", "rock"])},
#      "embedding": [0.1, 0.1, 0.1]},
# ]

# data = [
#     {"id": "1",
#      "meta": {"tags": ["pop", "rock"]},
#      "embedding": [0.1, 0.1, 0.1]},
#     {"id": "2",
#      "meta": {"tags": ["pop", "rock"]},
#      "embedding": [0.1, 0.1, 0.1]},
#     {"id": "3",
#      "meta": {"tags": ["pop", "rock"]},
#      "embedding": [0.1, 0.1, 0.1]},
# ]

data = [
    {"id": "1",
     "tags": ["pop", "rock"],
     "embedding": [0.1, 0.1, 0.1]},
    {"id": "2",
     "tags": ["pop", "rock"],
     "embedding": [0.1, 0.1, 0.1]},
    {"id": "3",
     "tags": ["pop", "rock"],
     "embedding": [0.1, 0.1, 0.1]},
]

# data = [
#     {"id": "1",
#      "meta": {"tags": [1, 2]},
#      "embedding": [0.1, 0.1, 0.1]},
#     {"id": "2",
#      "meta": {"tags": [1, 2]},
#      "embedding": [0.1, 0.1, 0.1]},
#     {"id": "3",
#      "meta": {"tags": [1, 2]},
#      "embedding": [0.1, 0.1, 0.1]},
# ]

docs = [Document.from_dict(d) for d in data]
docs

[Document(id=1, meta: {'tags': ['pop', 'rock']}, embedding: vector of size 3),
 Document(id=2, meta: {'tags': ['pop', 'rock']}, embedding: vector of size 3),
 Document(id=3, meta: {'tags': ['pop', 'rock']}, embedding: vector of size 3)]

In [76]:
writer.run(docs)

Document 1 has metadata fields with unsupported types: ['tags']. Supported types refer to Pymilvus DataType. The values of these fields will be discarded.
Document 2 has metadata fields with unsupported types: ['tags']. Supported types refer to Pymilvus DataType. The values of these fields will be discarded.
Document 3 has metadata fields with unsupported types: ['tags']. Supported types refer to Pymilvus DataType. The values of these fields will be discarded.


{'documents_written': 3}

In [53]:
from pandas.api.types import (
    infer_dtype,
    is_array_like,
    is_float,
    is_list_like,
    is_scalar,
)

In [63]:
myvar = ["a", "b"]
myvar = iter(["a", "b"])
is_list_like(data)

False